# Set Up

## Mount Google Drive

Ignore if not using Google Collab:

In [ ]:
from google.colab import drive

# mount google drive
drive.mount('/content/drive')
%cd /content/drive/My Drive
!git clone https://github.com/FranciscoLozCoding/cooling_with_code.git
%cd cooling_with_code
!git pull

## Import Libraries

In [ ]:
%%capture
%pip install shap

In [ ]:
# Supress Warnings
import warnings
warnings.filterwarnings('ignore')

#data science
import pandas as pd
import numpy as np
import shap
import pickle

#other
import os

#custom tools
from tools.environment import VALID_SPLIT, RANDOM_STATE, TARGET_VARIABLE
from tools.preprocess import apply_preprocessing_mixed_buffers, load_and_preprocess_data
from tools.explainer import Explainer
from tools.build_dataset import combine_buffer_datasets

## Load Model

Here we will load our best model.

In [ ]:
# get model
model_file = 'models/mixed_buffers_ResNet/mixed_buffers_ResNet_model.keras'
if os.path.exists(model_file):
  resnet_model = load_model(model_file)

# scaler
scaler_file = 'models/mixed_buffers_ResNet/mixed_buffers_standard_scaler.pkl'
with open(scaler_file, 'rb') as f:
  scaler = pickle.load(f)

## Import Datasets

Here we will import the dataset used by our best model.

In [ ]:
train_combined, test_combined = combine_buffer_datasets('data/train', 'data/test')
train_combined, test_combined = apply_preprocessing_mixed_buffers(train_combined, test_combined)

## Make Predictions

Here we will make predictions using our best model. SHAP will need them.

In [ ]:
# Remove lat and lon, for now will be combined with predicted UHI later for submission
lon = test_combined['Longitude'].values
lat = test_combined['Latitude'].values
test_dataset = test_combined.drop(columns=['Longitude', 'Latitude'])

#select features
test_dataset = test_dataset[selected_feature_names]

#scale
x_test = pd.DataFrame(scaler.transform(test_dataset),
                      columns=test_dataset.columns,
                      index=test_dataset.index).values

# Model prediction
y_pred = resnet_model.predict(x_test)

# In-Depth SHAP Analysis

This notebook is dedicated to performing an in-depth [**SHAP analysis**](https://shap.readthedocs.io/en/latest/) on our optimal predictive model, the **Mixed Buffer ResNet**. Although this model exhibited a slightly lower R-squared score compared to alternatives, it demonstrated reduced signs of overfitting and was more computationally efficient, with a significantly smaller file size (44 MB). Further details about the **Mixed Buffer ResNet** model, including its training process and initial evaluations, can be found in our [11_mixed_buffers_ResNet notebook](/11_mixed_buffers_RestNet.ipynb). While the previous notebook included a preliminary SHAP analysis, the present notebook aims to substantially expand upon those findings. Specifically, this comprehensive SHAP analysis will elucidate feature importance rankings, clarify the relationships between individual features and model predictions, and provide explanations for individual prediction outcomes. This analysis is crucial to gaining deeper insights into the key drivers influencing the Urban Heat Island phenomenon.



In [ ]:
# Initialize SHAP Explainer for keras model depending if shap_values were precomputed
shap_vals_file = 'data/test/mixed_buffers_ResNet_shap_vals.parquet'

if os.path.exists(shap_vals_file):
  shap_values = pd.read_parquet(shap_vals_file).values
else:
  shap_values = None

explainer = Explainer(
    model=resnet_model,
    explainer_type=shap.DeepExplainer,
    X=x_test,
    feature_names=test_dataset.columns,
    ref_data=x_test,
    shap_values=shap_values
)

# save SHAP values as Parquet so we don't have to run again
shap_df = pd.DataFrame(explainer.shap_values.values, columns=test_dataset.columns)
shap_df.to_parquet(shap_vals_file)

## Summary Plot

Here, we will see the most important features and how do features impact predictions. This is how to interpret the plot:

- Each **dot** represents a single data point from our dataset.
- **X-Axis (`SHAP Value`)**: The **impact** of a feature on the prediction.
- **Y-Axis (Feature Names)**: The **most important features** ranked from top to bottom.
- **Color (Red to Blue)**: The **feature value** (High = Red, Low = Blue).
  - **Red** = High feature value.
  - **Blue** = Low feature value.

1. **Feature Importance**  
   - Features are **ranked by importance** (most important at the top).
   - The **wider** the spread on the X-axis, the more that feature **influences predictions**.

2. **Direction of Influence**  
   - **Positive SHAP values (Right Side)**: Feature **increases** the prediction.
   - **Negative SHAP values (Left Side)**: Feature **decreases** the prediction.

3. **Effect of Feature Value (Color Gradient)**  
   - **Red (High Value)** trends to the **right**? High values **increase** predictions.
   - **Blue (Low Value)** trends to the **left**? Low values **decrease** predictions.

In [ ]:
explainer.summary_plot()

>TODO